# Modern Deep Learning Architectures for Time Series

The previous notebooks covered the foundational architectures: MLPs,
RNNs, LSTMs, and encoder-decoder models.  This notebook surveys the
**modern architectures** that represent the current state of the art
in deep learning for time series forecasting.

These architectures are *conceptual* -- we focus on understanding their
design principles and when to use them, rather than full implementations
(which require specialized libraries and large datasets).

**Contents:**
1. Temporal Convolutional Networks (TCN)
2. N-BEATS: Neural Basis Expansion Analysis
3. Temporal Fusion Transformer (TFT)
4. Foundation models: TimesFM, Chronos, Lag-Llama
5. When to use deep learning vs statistical methods
6. Decision framework

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import keras
from keras import layers

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. Temporal Convolutional Networks (TCN)

TCNs apply **1-D convolutions** to sequences instead of recurrence.
Two key innovations make them suitable for time series:

### Causal Convolutions

A causal convolution at time $t$ only uses inputs from time $\leq t$ --
it never looks into the future.  This is achieved by padding the input
on the left only.

### Dilated Convolutions

To capture long-range dependencies without stacking many layers,
TCNs use **dilated** convolutions that skip over inputs:

- Dilation $d=1$: standard convolution (adjacent inputs)
- Dilation $d=2$: skip every other input
- Dilation $d=4$: skip 3 inputs between each pair

The **receptive field** grows exponentially with depth:

$$
\text{Receptive field} = 1 + 2(k - 1)\sum_{i=0}^{L-1} d_i
$$

where $k$ is the kernel size, $L$ is the number of layers, and $d_i$
is the dilation factor at layer $i$ (typically $d_i = 2^i$).

In [ ]:
# Visualize dilated causal convolutions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, dilation in zip(axes, [1, 2, 4]):
    n_inputs = 16
    kernel_size = 3

    # Draw input nodes
    for i in range(n_inputs):
        ax.plot(i, 0, "o", color="tab:blue", markersize=8)

    # Draw which inputs contribute to the output at position 15
    output_pos = n_inputs - 1
    contributing = []
    for k in range(kernel_size):
        pos = output_pos - k * dilation
        if pos >= 0:
            contributing.append(pos)
            ax.plot([pos, output_pos], [0, 1], "r-", linewidth=1.5, alpha=0.7)
            ax.plot(pos, 0, "o", color="tab:red", markersize=10, zorder=5)

    ax.plot(output_pos, 1, "s", color="tab:green", markersize=12, zorder=5)

    ax.set_title(f"Dilation = {dilation}\n(span = {max(contributing) - min(contributing) if contributing else 0} steps)")
    ax.set_xlabel("Input position")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Input", "Output"])
    ax.set_xlim(-1, n_inputs)

plt.suptitle("Dilated Causal Convolutions (kernel size = 3)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Dilation=1: convolution sees 3 adjacent inputs.")
print("Dilation=4: convolution spans 9 time steps with only 3 weights.")
print("Stacking layers with d=1,2,4,8,... gives exponential receptive fields.")

In [ ]:
# Demonstrate a simple TCN-like model in Keras
# (Not a full TCN implementation, but shows the core idea)

def build_simple_tcn(input_length, n_features=1, n_filters=32, kernel_size=3):
    """Build a minimal TCN-style model using dilated causal Conv1D."""
    model = keras.Sequential([
        # Dilated causal convolutions with exponentially increasing dilation
        layers.Conv1D(n_filters, kernel_size, dilation_rate=1, padding="causal",
                      activation="relu", input_shape=(input_length, n_features)),
        layers.Conv1D(n_filters, kernel_size, dilation_rate=2, padding="causal",
                      activation="relu"),
        layers.Conv1D(n_filters, kernel_size, dilation_rate=4, padding="causal",
                      activation="relu"),
        layers.Conv1D(n_filters, kernel_size, dilation_rate=8, padding="causal",
                      activation="relu"),

        # Global average pooling -> Dense output
        layers.GlobalAveragePooling1D(),
        layers.Dense(1),
    ])
    return model

tcn_model = build_simple_tcn(input_length=24)
tcn_model.summary()

# Calculate receptive field
k = 3
dilations = [1, 2, 4, 8]
receptive_field = 1 + sum([(k - 1) * d for d in dilations])
print(f"\nReceptive field: {receptive_field} time steps")
print(f"With only {len(dilations)} layers and kernel_size={k}!")

### TCN vs LSTM

| Property | TCN | LSTM |
|----------|-----|------|
| **Parallelizable** | Yes (all positions at once) | No (sequential by design) |
| **Receptive field** | Fixed (set by architecture) | Theoretically infinite |
| **Training speed** | Faster (parallelism) | Slower (sequential) |
| **Memory** | $O(\text{layers} \times \text{filters})$ | $O(\text{hidden size})$ |
| **Typical use** | When speed matters, fixed-length inputs | Variable-length, long dependencies |

---
## 2. N-BEATS: Neural Basis Expansion Analysis

**N-BEATS** (Oreshkin et al., 2019) is a *pure deep learning* architecture
for time series -- no recurrence, no convolutions, just fully connected
layers.  It achieves state-of-the-art results on many benchmarks.

### Architecture

N-BEATS organizes computation into **stacks** of **blocks**:

1. Each **block** takes an input (the lookback window or a residual)
   and produces two outputs:
   - **Backcast**: the block's reconstruction of the *past* (input)
   - **Forecast**: the block's prediction of the *future*

2. The **residual** (input minus backcast) is passed to the next block

3. Final forecast = sum of all blocks' forecasts

$$
\hat{\mathbf{x}}_\ell^{(b)} = \sum_{k} \theta_k^{(b)} \mathbf{v}_k^{(b)} \quad \text{(backcast)}
$$
$$
\hat{\mathbf{y}}_\ell^{(f)} = \sum_{k} \theta_k^{(f)} \mathbf{v}_k^{(f)} \quad \text{(forecast)}
$$

### Interpretable Configuration

In the interpretable version, the basis vectors $\mathbf{v}_k$ are
fixed to meaningful patterns:
- **Trend stack**: polynomial basis (linear, quadratic, etc.)
- **Seasonality stack**: Fourier basis (sines and cosines)

In [ ]:
# Visualize the N-BEATS block concept
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

np.random.seed(42)
t_back = np.arange(24)
t_fore = np.arange(24, 36)

# Simulate an input signal
input_signal = np.sin(2 * np.pi * t_back / 12) + 0.05 * t_back + np.random.randn(24) * 0.2

# Block 1: captures trend
trend_backcast = 0.05 * t_back
trend_forecast = 0.05 * t_fore
residual_1 = input_signal - trend_backcast

# Block 2: captures seasonality from residual
season_backcast = np.sin(2 * np.pi * t_back / 12)
season_forecast = np.sin(2 * np.pi * t_fore / 12)

# Block 1
axes[0].plot(t_back, input_signal, "b-", label="Input", alpha=0.7)
axes[0].plot(t_back, trend_backcast, "r--", label="Backcast (trend)", linewidth=2)
axes[0].plot(t_fore, trend_forecast, "r:", label="Forecast (trend)", linewidth=2)
axes[0].set_title("Block 1: Trend")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Block 2 (receives residual)
axes[1].plot(t_back, residual_1, "b-", label="Residual", alpha=0.7)
axes[1].plot(t_back, season_backcast, "g--", label="Backcast (seasonal)", linewidth=2)
axes[1].plot(t_fore, season_forecast, "g:", label="Forecast (seasonal)", linewidth=2)
axes[1].set_title("Block 2: Seasonality (from residual)")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Combined forecast
combined_forecast = trend_forecast + season_forecast
actual_future = np.sin(2 * np.pi * t_fore / 12) + 0.05 * t_fore

axes[2].plot(t_back, input_signal, "b-", alpha=0.4, label="History")
axes[2].plot(t_fore, actual_future, "k-", linewidth=2, label="Actual")
axes[2].plot(t_fore, combined_forecast, "m--", linewidth=2,
             label="Combined forecast")
axes[2].set_title("Combined: Trend + Seasonality")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel("Time")

plt.suptitle("N-BEATS: Backcast/Forecast Decomposition", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Each block explains part of the signal (backcast) and predicts (forecast).")
print("The residual is passed to the next block for further decomposition.")

### Key Properties of N-BEATS

- **Pure deep learning**: no hand-crafted features, no explicit seasonal
  decomposition (the generic version)
- **Interpretable version**: trend + seasonality stacks with structured
  basis functions
- **Doubly residual**: residual connections at both the block level
  (input minus backcast) and the stack level
- **State-of-the-art** on M3 and M4 competition datasets
- Available in libraries: [Darts](https://unit8co.github.io/darts/),
  [NeuralForecast](https://nixtla.github.io/neuralforecast/)

---
## 3. Temporal Fusion Transformer (TFT)

The **TFT** (Lim et al., 2021) is an attention-based architecture
specifically designed for multi-horizon time series forecasting.  It
is one of the most capable architectures for real-world forecasting
problems.

### Key Innovations

1. **Variable Selection Networks**: automatically learn which input
   features are most important at each time step

2. **Static Covariate Encoders**: handle time-invariant features
   (e.g., store ID, product category)

3. **Temporal Processing**: combines LSTM (for local patterns) with
   multi-head self-attention (for long-range dependencies)

4. **Gated Residual Networks (GRN)**: gated skip connections that
   control information flow

5. **Quantile Outputs**: produces prediction intervals, not just
   point forecasts

### Input Types TFT Can Handle

| Input type | Description | Example |
|-----------|-------------|----------|
| **Past observed** | Historical target values | Sales in past 30 days |
| **Past known** | Historical covariates (known) | Past promotions, holidays |
| **Future known** | Covariates known in advance | Planned promotions, day of week |
| **Static** | Time-invariant metadata | Store ID, product category |

In [ ]:
# Visualize TFT architecture conceptually
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

# Boxes for components
components = [
    (1, 1, 3, 1.2, "Input Embedding\n& Variable Selection", "tab:blue"),
    (6, 1, 3, 1.2, "Static Covariate\nEncoders", "tab:cyan"),
    (1, 3, 3, 1.2, "LSTM Encoder\n(local patterns)", "tab:orange"),
    (6, 3, 3, 1.2, "LSTM Decoder\n(local patterns)", "tab:orange"),
    (3.5, 5.5, 3, 1.2, "Multi-Head\nSelf-Attention", "tab:red"),
    (3.5, 7.5, 3, 1.2, "Quantile\nOutput", "tab:green"),
]

for x, y, w, h, label, color in components:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                    facecolor=color, alpha=0.3, edgecolor=color)
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center",
            fontsize=10, fontweight="bold")

# Arrows
arrows = [
    (2.5, 2.2, 0, 0.8),    # input -> encoder
    (7.5, 2.2, 0, 0.8),    # static -> decoder
    (4, 4.2, 0, 1.3),      # encoder -> attention
    (6, 4.2, 0, 1.3),      # decoder -> attention
    (5, 6.7, 0, 0.8),      # attention -> output
]
for x, y, dx, dy in arrows:
    ax.annotate("", xy=(x + dx, y + dy), xytext=(x, y),
               arrowprops=dict(arrowstyle="->", lw=2, color="grey"))

ax.set_title("Temporal Fusion Transformer (TFT) Architecture", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("TFT combines the best of LSTMs (local patterns) and attention (long-range).")
print("Variable selection automatically identifies the most important inputs.")

### Self-Attention for Time Series

The self-attention mechanism allows each time step to attend to *all
other* time steps, weighted by relevance:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

where $Q$ (query), $K$ (key), $V$ (value) are linear projections of
the input.  This lets the model learn which past time steps are most
relevant for predicting each future step -- for example, learning
that "the same month last year" is highly informative for seasonal data.

In [ ]:
# Demonstrate multi-head attention in Keras
# This is the core building block of TFT and Transformers

seq_len = 24
d_model = 32

attention_layer = layers.MultiHeadAttention(
    num_heads=4,
    key_dim=d_model // 4,
)

# Dummy input: (batch, seq_len, d_model)
dummy = np.random.randn(1, seq_len, d_model).astype("float32")
output = attention_layer(dummy, dummy)  # self-attention: query=key=value

print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {output.shape}")
print(f"Parameters:   {attention_layer.count_params():,}")
print()
print("Multi-head attention lets each time step attend to all others.")
print("4 heads with key_dim=8 gives each head a different 'perspective'.")

---
## 4. Foundation Models for Time Series

Inspired by the success of large language models (GPT, BERT), several
**foundation models** have been developed specifically for time series.
These are pre-trained on massive collections of time series and can be
used zero-shot (no fine-tuning) or with minimal adaptation.

### TimesFM (Google, 2024)

- **Architecture**: Decoder-only Transformer (similar to GPT)
- **Pre-trained on**: 100B+ time points from Google Trends, Wikipedia,
  and synthetic data
- **Approach**: treats time series as a "language" of numerical patches
- **Key feature**: zero-shot forecasting -- no training on your data

### Chronos (Amazon, 2024)

- **Architecture**: T5-based encoder-decoder
- **Pre-trained on**: diverse public time series datasets + synthetic data
- **Approach**: tokenizes real-valued time series into discrete bins,
  then uses standard language model training
- **Key feature**: probabilistic forecasts via token sampling

### Lag-Llama (ServiceNow, 2023)

- **Architecture**: LLaMA-style decoder-only Transformer
- **Pre-trained on**: 27 diverse time series datasets
- **Approach**: uses lag features (e.g., value at $t-1$, $t-7$, $t-12$)
  as input tokens
- **Key feature**: designed for probabilistic forecasting with
  Student's t distribution output

In [ ]:
# Comparison table of foundation models
foundation_models = pd.DataFrame({
    "Model": ["TimesFM", "Chronos", "Lag-Llama"],
    "Organization": ["Google", "Amazon", "ServiceNow"],
    "Architecture": ["Decoder-only Transformer", "T5 Encoder-Decoder", "LLaMA-style Decoder"],
    "Tokenization": ["Numerical patches", "Discrete bins", "Lag features"],
    "Zero-shot": ["Yes", "Yes", "Yes (after fine-tune)"],
    "Probabilistic": ["Point + intervals", "Yes (sampling)", "Yes (Student-t)"],
    "Open Source": ["Yes", "Yes", "Yes"],
})

print(foundation_models.to_string(index=False))

In [ ]:
# Visualize the foundation model approach
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Traditional approach
ax = axes[0]
ax.set_title("Traditional: Train per Dataset", fontsize=12)
datasets = ["Dataset A", "Dataset B", "Dataset C"]
colors = ["tab:blue", "tab:orange", "tab:green"]
for i, (ds, c) in enumerate(zip(datasets, colors)):
    ax.barh(i, 1, color=c, alpha=0.5, edgecolor=c)
    ax.text(0.5, i, f"{ds} -> Model {chr(65+i)}", ha="center", va="center", fontsize=10)
ax.set_yticks([])
ax.set_xticks([])
ax.set_xlim(-0.1, 1.5)
ax.text(0.5, -0.7, "One model per dataset\nSmall data = poor performance",
        ha="center", fontsize=9, style="italic")

# Foundation model approach
ax = axes[1]
ax.set_title("Foundation: Pre-train Once, Use Everywhere", fontsize=12)
for i, (ds, c) in enumerate(zip(["100B+ time points", "Diverse domains", "Synthetic data"], colors)):
    ax.barh(i + 1.5, 1, color=c, alpha=0.3, edgecolor=c)
    ax.text(0.5, i + 1.5, ds, ha="center", va="center", fontsize=10)
ax.barh(0, 1.2, color="tab:red", alpha=0.5, edgecolor="tab:red")
ax.text(0.6, 0, "One Foundation Model -> Any Dataset", ha="center", va="center",
        fontsize=10, fontweight="bold")
ax.set_yticks([])
ax.set_xticks([])
ax.set_xlim(-0.1, 1.7)
ax.text(0.6, -0.7, "Pre-trained on massive data\nZero-shot or minimal fine-tuning",
        ha="center", fontsize=9, style="italic")

plt.tight_layout()
plt.show()

### When Foundation Models Shine

- **Cold start**: no historical data for a new product/store
- **Quick prototyping**: get a reasonable forecast without any training
- **Cross-domain transfer**: patterns learned from one domain help another

### Current Limitations

- **Not always best**: on well-studied datasets with enough data,
  fine-tuned task-specific models often still win
- **Univariate focus**: most current models handle univariate series;
  multivariate support is still developing
- **Inference cost**: large models require GPU for reasonable speed
- **Black box**: even less interpretable than standard deep learning

---
## 5. When to Use Deep Learning vs Statistical Methods

A critical practical question: **when should you reach for deep learning
instead of ARIMA, ETS, or Prophet?**

The honest answer: **statistical methods are hard to beat on individual
short series.**  This has been demonstrated repeatedly in forecasting
competitions (M3, M4, M5).

### Why Statistical Methods Win on Short Series

1. **Parameter efficiency**: ARIMA/ETS have 3-7 parameters;
   neural networks have thousands or millions
2. **Inductive bias**: statistical models encode strong assumptions
   about time series structure (trend, seasonality, autocorrelation)
   that act as powerful regularizers
3. **No overfitting risk**: with so few parameters, statistical methods
   rarely overfit even on 50-100 observations
4. **Interpretability**: you can inspect components, residuals, and
   understand *why* the forecast looks the way it does

### When Deep Learning Wins

1. **Large-scale forecasting**: thousands of related time series
   (e.g., all products in a retail chain), where cross-series learning
   is possible
2. **Complex patterns**: non-linear interactions, regime changes,
   high-dimensional inputs
3. **Rich covariates**: many external features (promotions, weather,
   economic indicators)
4. **Multi-step horizons**: direct multi-step forecasts via seq2seq
5. **Sufficient data**: hundreds or thousands of observations per series

In [ ]:
# Decision table
decision = pd.DataFrame({
    "Scenario": [
        "Single short series (< 200 obs)",
        "Single long series (> 1000 obs)",
        "Many related series (cross-learning)",
        "Rich external covariates",
        "Non-linear / complex patterns",
        "Need interpretability",
        "Need fast iteration",
        "Cold start / no history",
        "Multi-step forecast needed",
        "Production system at scale",
    ],
    "Recommended": [
        "Statistical (ETS/ARIMA)",
        "Either (test both)",
        "Deep Learning (TFT, N-BEATS)",
        "Deep Learning (TFT)",
        "Deep Learning",
        "Statistical or Prophet",
        "Statistical (faster dev cycle)",
        "Foundation Model (Chronos, TimesFM)",
        "Seq2Seq or Direct DL",
        "Depends on data volume",
    ],
    "Why": [
        "DL overfits; statistical models have better inductive bias",
        "Enough data for DL, but statistical may still be competitive",
        "DL can learn shared patterns across series",
        "Statistical models cannot easily incorporate many covariates",
        "Statistical models assume linearity",
        "Component decomposition is transparent",
        "No GPU needed, minutes vs hours",
        "Pre-trained on diverse data, zero-shot capable",
        "Avoids error accumulation of recursive approaches",
        "Weigh data volume against engineering complexity",
    ],
})

# Display with wrapping
pd.set_option("display.max_colwidth", 60)
print(decision.to_string(index=False))
pd.reset_option("display.max_colwidth")

---
## 6. Architecture Decision Framework

In [ ]:
# Visualize the decision framework
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis("off")

# Decision tree (simplified)
nodes = {
    "start": (5, 9, "How much data?"),
    "small": (2, 7, "< 200 obs\nper series"),
    "large": (8, 7, "> 200 obs\nper series"),
    "stat": (2, 5, "ETS / ARIMA\n/ Prophet"),
    "multi": (6, 5, "Many related\nseries?"),
    "single_dl": (5, 3, "LSTM\n/ TCN"),
    "cross": (8, 5, "Covariates?"),
    "nbeats": (7, 3, "N-BEATS\n(univariate)"),
    "tft": (9, 3, "TFT\n(multivariate)"),
    "cold": (11, 7, "No history?\nFoundation Model"),
}

for key, (x, y, label) in nodes.items():
    color = "lightyellow" if key in ["stat", "single_dl", "nbeats", "tft", "cold"] else "lightblue"
    bbox = dict(boxstyle="round,pad=0.5", facecolor=color, edgecolor="grey")
    ax.text(x, y, label, ha="center", va="center", fontsize=10,
            fontweight="bold", bbox=bbox)

# Arrows
arrow_pairs = [
    ("start", "small", "Few"),
    ("start", "large", "Many"),
    ("small", "stat", ""),
    ("large", "multi", "Yes"),
    ("large", "cross", "No"),
    ("multi", "single_dl", "No"),
    ("multi", "nbeats", "Univariate"),
    ("cross", "nbeats", "No"),
    ("cross", "tft", "Yes"),
]

for src, dst, label in arrow_pairs:
    x1, y1, _ = nodes[src]
    x2, y2, _ = nodes[dst]
    ax.annotate("", xy=(x2, y2 + 0.5), xytext=(x1, y1 - 0.5),
               arrowprops=dict(arrowstyle="->", lw=1.5, color="grey"))
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx - 0.3, my + 0.1, label, fontsize=8, color="grey", style="italic")

ax.set_xlim(0, 12)
ax.set_ylim(1.5, 10)
ax.set_title("Architecture Decision Framework", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

---
## Architecture Summary Table

In [ ]:
summary = pd.DataFrame({
    "Architecture": ["MLP", "LSTM/GRU", "Seq2Seq", "TCN", "N-BEATS", "TFT", "Foundation"],
    "Type": ["Feedforward", "Recurrent", "Encoder-Decoder", "Convolutional",
             "Feedforward", "Attention", "Pre-trained"],
    "Strengths": [
        "Simple, fast",
        "Sequential memory",
        "Multi-step output",
        "Parallel, fast",
        "Pure DL, interpretable config",
        "Covariates, attention",
        "Zero-shot, transfer",
    ],
    "Limitations": [
        "No temporal structure",
        "Slow training, vanishing gradients",
        "Complex, needs more data",
        "Fixed receptive field",
        "Univariate only",
        "Complex, slow",
        "Black box, GPU needed",
    ],
    "Min Data": ["~100", "~200", "~500", "~200", "~200", "~1000", "0 (zero-shot)"],
})

pd.set_option("display.max_colwidth", 40)
print(summary.to_string(index=False))
pd.reset_option("display.max_colwidth")

---
## Available Libraries

| Library | Architectures | Install |
|---------|--------------|----------|
| [Keras](https://keras.io) | MLP, LSTM, GRU, Seq2Seq, TCN (manual) | `pip install keras` |
| [Darts](https://unit8co.github.io/darts/) | N-BEATS, TFT, TCN, LSTM, + statistical | `pip install darts` |
| [NeuralForecast](https://nixtla.github.io/neuralforecast/) | N-BEATS, TFT, PatchTST, + more | `pip install neuralforecast` |
| [GluonTS](https://ts.gluon.ai/) | DeepAR, TFT, + many | `pip install gluonts` |
| [Chronos](https://github.com/amazon-science/chronos-forecasting) | Chronos foundation model | `pip install chronos-forecasting` |
| [TimesFM](https://github.com/google-research/timesfm) | TimesFM foundation model | `pip install timesfm` |

---
## Summary

- **TCNs** use dilated causal convolutions for parallel, efficient
  sequence processing with exponentially growing receptive fields

- **N-BEATS** is a pure deep learning architecture that decomposes
  forecasts into backcast/forecast stacks, achieving state-of-the-art
  results with no hand-crafted features

- **TFT** is an attention-based architecture that handles multiple
  input types (past observed, future known, static covariates) and
  produces interpretable, probabilistic multi-horizon forecasts

- **Foundation models** (TimesFM, Chronos, Lag-Llama) are pre-trained
  on massive datasets and can forecast zero-shot or with minimal
  fine-tuning

- **Statistical methods are hard to beat on individual short series.**
  Deep learning shines on large-scale, complex, multivariate problems
  with sufficient data

- **Always start with a simple baseline** (seasonal naive, ETS, ARIMA)
  and only move to deep learning if the baseline is insufficient and
  you have enough data to justify the complexity

In [ ]:
print("This concludes Chapter 12: Deep Learning for Time Series.")
print()
print("Key takeaway: deep learning is a powerful tool, but not always")
print("the right one. Match the method to the problem.")